# **Atelier — Préparation des données images**

**Objectif :** construire un jeu de données d'images propre et homogène, prêt pour un modèle de Machine Learning ou de Deep Learning.

Le dataset contient les classes :
- **cardboard**
- **glass**
- **metal**
- **paper**
- **plastic**
- **trash**

## **Structure attendue**

```text
atelier_prepa_donnees_images/
├── notebooks/
│   └── atelier_prepa_donnees_images.ipynb
├── reports/
│   └── audit_images.csv
└── data/
    ├── raw/
    │   ├── cardboard/
    │   ├── glass/
    │   ├── metal/
    │   ├── paper/
    │   ├── plastic/
    │   └── trash/
    └── cleaned/
        ├── cardboard/
        ├── glass/
        ├── metal/
        ├── paper/
        ├── plastic/
        └── trash/
```

Le dossier **cleaned** est destiné aux données modifiées ; le dataset original reste dans **raw**.


# **Partie 1 — Exploration du dataset**

Pour chaque image on recupére : son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l'écart-type de ses pixels, son nombre de canaux et sa taille, tout en prenant en charge les fichiers corrompus.

In [6]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

data_dir = Path("../data/raw")
reports_dir = Path("../reports")

print("Dossier des données :", data_dir)
print("Existe :", data_dir.exists())


Dossier des données : ..\data\raw
Existe : True


In [7]:
classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])

print("Classes détectées :")
for classe in classes:
    print("-", classe)


Classes détectées :
- cardboard
- glass
- metal
- paper
- plastic
- trash


In [8]:
images_info = []

for classe in classes:
    classe_dir = data_dir / classe

    for fichier in sorted(classe_dir.iterdir()):
        if not fichier.is_file():
            continue

        try:
            with Image.open(fichier) as image:
                image.verify()

            with Image.open(fichier) as image:
                largeur, hauteur = image.size
                format_image = image.format
                mode = image.mode
                pixels = np.array(image)

                if pixels.ndim == 2:
                    nombre_canaux = 1
                else:
                    nombre_canaux = pixels.shape[2]

                ecart_type = float(pixels.std())
                taille = fichier.stat().st_size

                images_info.append({
                    "nom": fichier.name,
                    "classe": classe,
                    "format": format_image,
                    "mode": mode,
                    "largeur": largeur,
                    "hauteur": hauteur,
                    "ecart_type_pixels": ecart_type,
                    "nombre_canaux": nombre_canaux,
                    "taille_octets": taille,
                    "corrompue": False,
                    "chemin": str(fichier)
                })

        except Exception:
            images_info.append({
                "nom": fichier.name,
                "classe": classe,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type_pixels": None,
                "nombre_canaux": None,
                "taille_octets": fichier.stat().st_size,
                "corrompue": True,
                "chemin": str(fichier)
            })

df_images = pd.DataFrame(images_info)

print("Nombre total de fichiers analysés :", len(df_images))
df_images.head()


Nombre total de fichiers analysés : 1032


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille_octets,corrompue,chemin
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,..\data\raw\cardboard\cardboard1.jpg
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,..\data\raw\cardboard\cardboard10.jpg
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,..\data\raw\cardboard\cardboard100.jpg
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,..\data\raw\cardboard\cardboard101.jpg
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,..\data\raw\cardboard\cardboard102.jpg


In [9]:
print("Dimensions du tableau :", df_images.shape)
print("Nombre d'images corrompues :", int(df_images["corrompue"].sum()))

df_images

Dimensions du tableau : (1032, 11)
Nombre d'images corrompues : 6


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille_octets,corrompue,chemin
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,..\data\raw\cardboard\cardboard1.jpg
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,..\data\raw\cardboard\cardboard10.jpg
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,..\data\raw\cardboard\cardboard100.jpg
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,..\data\raw\cardboard\cardboard101.jpg
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,..\data\raw\cardboard\cardboard102.jpg
...,...,...,...,...,...,...,...,...,...,...,...
1027,trash50.jpg,trash,JPEG,RGB,512.0,384.0,41.780027,3.0,13563,False,..\data\raw\trash\trash50.jpg
1028,trash6.jpg,trash,JPEG,RGB,512.0,384.0,37.391702,3.0,10534,False,..\data\raw\trash\trash6.jpg
1029,trash7.jpg,trash,JPEG,RGB,512.0,384.0,41.297376,3.0,8011,False,..\data\raw\trash\trash7.jpg
1030,trash8.jpg,trash,JPEG,RGB,512.0,384.0,47.635953,3.0,16946,False,..\data\raw\trash\trash8.jpg


In [10]:
reports_dir.mkdir(parents=True, exist_ok=True)

audit_path = reports_dir / "audit_images.csv"
df_images.to_csv(audit_path, index=False)

print("Audit enregistré dans :", audit_path)

Audit enregistré dans : ..\reports\audit_images.csv


 # **Partie 2 — Détecter les images corrompues**

On crée une fonction dédiée à la détection des images qui ne peuvent pas être ouvertes ou vérifiées correctement.


In [11]:
from pathlib import Path
from PIL import Image

def detecter_image_corrompue(chemin):
    try:
        with Image.open(chemin) as image:
            image.verify()
        return False
    except Exception:
        return True

In [12]:
images_corrompues = []

for classe in classes:
    classe_dir = data_dir / classe

    for fichier in sorted(classe_dir.iterdir()):
        if fichier.is_file() and detecter_image_corrompue(fichier):
            images_corrompues.append({
                "nom": fichier.name,
                "classe": classe,
                "chemin": str(fichier)
            })

df_corrompues = pd.DataFrame(images_corrompues)

print("Nombre d'images corrompues :", len(df_corrompues))
df_corrompues

Nombre d'images corrompues : 6


,nom,classe,chemin
0,cardboard83.jpg,cardboard,..\data\raw\cardboard\cardboard83.jpg
1,glass74.jpg,glass,..\data\raw\glass\glass74.jpg
2,metal48.jpg,metal,..\data\raw\metal\metal48.jpg
3,paper213.jpg,paper,..\data\raw\paper\paper213.jpg
4,plastic13.jpg,plastic,..\data\raw\plastic\plastic13.jpg
5,trash3.jpg,trash,..\data\raw\trash\trash3.jpg


# **Partie 3 — Détecter les images vides**

Une image est considérée comme problématique lorsqu'elle est :
- entièrement noire ;
- entièrement blanche ;
- ou lorsque ses pixels présentent très peu de variation.

On utilise ici l'écart-type des pixels comme mesure de variation.

In [13]:
from PIL import Image
import numpy as np

def detecter_image_vide(chemin, seuil_variation=1.0):
    try:
        with Image.open(chemin) as image:
            pixels = np.array(image.convert("RGB"))

        ecart_type = pixels.std()

        est_noire = np.all(pixels == 0)
        est_blanche = np.all(pixels == 255)
        tres_peu_de_variation = ecart_type < seuil_variation

        return est_noire or est_blanche or tres_peu_de_variation

    except Exception:
        return False


In [14]:
images_vides = []

for classe in classes:
    classe_dir = data_dir / classe

    for fichier in sorted(classe_dir.iterdir()):
        if fichier.is_file() and not detecter_image_corrompue(fichier):
            if detecter_image_vide(fichier):
                images_vides.append({
                    "nom": fichier.name,
                    "classe": classe,
                    "chemin": str(fichier)
                })

df_vides = pd.DataFrame(images_vides)

print("Nombre d'images vides ou quasi vides :", len(df_vides))
df_vides

Nombre d'images vides ou quasi vides : 2


,nom,classe,chemin
0,image-noire-512x384.png,glass,..\data\raw\glass\image-noire-512x384.png
1,image-noire-512x384.png,metal,..\data\raw\metal\image-noire-512x384.png


# **Partie 4 — Détecter les différences de résolution**

## **4.1 Résolution minimale, maximale, fréquente et nombre d'images par résolution**

La résolution est représentée ici par le couple `(largeur, hauteur)`.

In [15]:
resolutions = []

for classe in classes:
    classe_dir = data_dir / classe

    for fichier in sorted(classe_dir.iterdir()):
        if not fichier.is_file():
            continue

        try:
            with Image.open(fichier) as image:
                largeur, hauteur = image.size

            resolutions.append({
                "nom": fichier.name,
                "classe": classe,
                "largeur": largeur,
                "hauteur": hauteur,
                "resolution": f"{largeur}x{hauteur}",
                "chemin": str(fichier)
            })

        except Exception:
            continue

df_resolutions = pd.DataFrame(resolutions)

print("Résolution minimale :")
print(df_resolutions[["largeur", "hauteur"]].min())

print("\nRésolution maximale :")
print(df_resolutions[["largeur", "hauteur"]].max())

print("\nRésolutions les plus fréquentes :")
print(df_resolutions["resolution"].value_counts().head(10))

Résolution minimale :
largeur    32
hauteur    32
dtype: int64

Résolution maximale :
largeur    512
hauteur    384
dtype: int64

Résolutions les plus fréquentes :
resolution
512x384    1013
32x32         5
48x32         4
40x40         4
Name: count, dtype: int64


In [16]:
images_trop_petites = df_resolutions[
    (df_resolutions["largeur"] < 64) |
    (df_resolutions["hauteur"] < 64)
]

print("Nombre d'images sous 64x64 :", len(images_trop_petites))

images_trop_petites

Nombre d'images sous 64x64 : 13


,nom,classe,largeur,hauteur,resolution,chemin
20,cardboard117.jpg,cardboard,48,32,48x32,..\data\raw\cardboard\cardboard117.jpg
77,cardboard22.jpg,cardboard,32,32,32x32,..\data\raw\cardboard\cardboard22.jpg
133,cardboard70.jpg,cardboard,40,40,40x40,..\data\raw\cardboard\cardboard70.jpg
170,glass100.jpg,glass,40,40,40x40,..\data\raw\glass\glass100.jpg
228,glass15.jpg,glass,48,32,48x32,..\data\raw\glass\glass15.jpg
267,glass21.jpg,glass,32,32,32x32,..\data\raw\glass\glass21.jpg
269,glass23.jpg,glass,32,32,32x32,..\data\raw\glass\glass23.jpg
381,metal121.jpg,metal,48,32,48x32,..\data\raw\metal\metal121.jpg
411,metal2.jpg,metal,32,32,32x32,..\data\raw\metal\metal2.jpg
419,metal26.jpg,metal,40,40,40x40,..\data\raw\metal\metal26.jpg
